In [21]:
%load_ext autoreload
%autoreload 2

In [166]:
import os
import re
import pymupdf
import pymupdf4llm
import pathlib as pl
import itertools
from IPython.display import display, Markdown

from dotenv import load_dotenv
# noinspection trailing-semicolon
load_dotenv();  # Suppress output

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, MarkdownTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableConfig

from tqdm.auto import tqdm

from risk_pipeline.models import ExtractionResult
from IPython.core.magic import register_cell_magic

In [167]:
@register_cell_magic
def skip(line, cell):
    message = f"skipping: {line.strip()}" if line.strip() else "skipping"
    print(message)

needed_env = ["OPENAI_API_KEY"]
for e in needed_env: assert os.getenv(e), f"{e} is not set. Please check your .env file or environment variables."
display(Markdown('**<span style="color:green">Ready to go</span>**'))

**<span style="color:green">Ready to go</span>**

In [12]:
case_path = pl.Path.cwd() / "doc/Case_GenAIandNLPEngineer.pdf"
md_case = pymupdf4llm.to_markdown(case_path)
display(Markdown(md_case))

# Case Study: Building a Structured Risk Intelligence pipeline 

The following case study is intended to measure your ability to understand, analyse, solve and advise on a business problem. We will use it to assess how you approach the problem, propose and construct a reasonable solution and layout a strategy for deployment in production including potential future improvements. We are interested in discussing the proposed solution at a high-level from a stakeholder engagement point of view, as well as at a deep technical level. 

As time for completing the task is very limited (~4–8 hours), we suggest that all attempts (even unsuccessful ones) are kept for discussion. We are interested in your train-of-thought process, how that is put into use in your technical solution and how you think about building a real product, more than in the level of performance of the solution itself. 

We expect you to use AI coding assistants (Cursor, Copilot, Claude Code, your IDE's built-in assistant, whatever you normally use). The walkthrough afterwards is where we'll calibrate your understanding, so reason as you build. 

## The product you're joining 

You're joining a small team at a research firm that sells structured risk intelligence to asset managers and compliance teams. Today our analysts read 200+ corporate annual reports per quarter to track each company's principal risks: how they're disclosed, how categorization shifts year-over-year and which emerging risks appear across a portfolio. The work is slow, expensive and inconsistent: two analysts reading the same report often surface different risks. 

We're building a **structured, queryable risk database** that turns those reports into something analysts and clients can search, filter and trend over time. Example questions the database should be able to answer: 

- _"What are the top enterprise risks facing Vestas, and how frequently are they reviewed at Board/Executive level?”_ 

- _"What emerging risks have been newly elevated?"_ 

- _“Show me every renewable-energy company that lists cyber security as a principal risk"_ 

Your job in this case: build the pipeline that turns one report into structured data and do it well. 

## The task 

We're giving you Vestas's Annual Report (PDF, ~190 pages). Build a small pipeline in python that: 

1. Ingests the PDF. 

2. Extracts the company's principal risks. For each risk, produce: 

   - a. A short title 

   - b. A 2-3 sentence description 

   - c. A category (e.g. financial, operational, regulatory, market, climate, cyber, supply chain) 

   - d. The section and page where you found it 

   - e. Any stated mitigation action 

Classification: Confidential 

3. Returns the result as a structured object (JSON, Pydantic, your choice of schema language). 

4. Evaluates itself. Build a small eval (a golden set + at least one evaluator) that would catch a regression, for example by swapping in a weaker model, breaking the parser, or changing a prompt. 

You may choose to structure your pipeline as a single step or multiple steps/tools. We are interested in how you decompose the problem and why (e.g. parsing vs extraction, single-pass vs multi-step). Considering the time available for the case, implement core part of the solution (your choice) while ready to provide a walkthrough of how the technical solution will be built into a real product. 

## Scope 

Focus on these sections of the PDF: 

- Risk management (p.50-51) 

- Material impacts, risks, and opportunities (p.71-74) 

You may also pull from Cyber Security (p.118) and Climate Change (p.85-92). You decide what parts of the PDF to ingest and you do not need to process all 190 pages. 

## Think product, not just task 

Before you write code, write a short PLAN.md (~½ page). Situate this single-report pipeline inside the broader product: 

- Who is the user, and how will they consume the output? 

- Roughly what does the product surface look like (batch API, interactive UI, alerting, …)? 

- What are you optimizing for in this first slice: correctness, coverage, cost, throughput? 

- What assumptions are you making? 

- What are you explicitly _not_ solving here but would tackle next? 

This isn't a polished product spec. It's a ~10-minute think-through that should visibly inform every implementation decision you make next. 

## Deliverables 

- PLAN.md: product context, user, optimization target, assumptions, what's deferred. 

- Working code: any stack, any model provider. Be ready to explain why you chose what you chose. 

- DESIGN.md (~1 page): your stack, parsing approach, your decomposition of the work, and key trade-offs. The trade-offs should trace back to choices in PLAN.md. Include considerations for scaling (latency, cost or throughput) if this were run across many reports 

- Eval artifacts: your golden set, the evaluator(s), and at least one sample run catching a realistic failure mode 

- STRETCH.md (short): what you'd build next if you had another week. 

Classification: Confidential 

## Walkthrough format (45 min total) 

After we receive your submission, we'll schedule a 60-minute walkthrough. 

- 30 minutes are yours. You drive. Prepare a focused walkthrough of the parts of your solution that _you_ think matter most. You won't have time to cover everything and deliberately choosing what to emphasize (and what to skip) is part of what we're listening for. Lead us through your reasoning, not just your code. We'll stay quiet unless we need a quick clarification. 

- 20 minutes for Q&A. We'll follow up on what you showed, probe areas you skipped and depending on role level, discuss how you'd evolve the solution at scale. 

- 10 minutes for your Questions. The final few minutes are yours for questions to us. 

## Constraints & ground rules 

- Time budget: ~4–8 hours of focused work. We assess what you got done _and_ how you reasoned about your cuts. 

- AI coding tools are allowed and expected. The walkthrough will probe understanding deeply enough that copy-paste alone won't pass. 

- No stack adherence. You're not graded on framework or provider choice. You're graded on outcomes and reasoning. 

- No starter code. You choose the project layout, dependencies, and structure. These choices are part of the task and what we evaluate. 

- Hybrid approaches welcome. Combining classical NLP (NER, regex, sentence segmentation, traditional information extraction) with LLM calls is fully encouraged. We're hiring for judgment about _when_ to reach for an LLM, not for LLM mastery alone. 

## Practicalities 

- Use Vestas Annual Report PDF shared with you 

- Submission: Please send a zip or git repo link 

- Questions during the take-home: If anything in this brief is genuinely blocking you, reach out. We'd rather you ask than be stuck. 

Good luck. We're looking forward to seeing how you think. 

Classification: Confidential 



# Parsing and extraction

In [120]:
report_path = pl.Path.cwd() / "data/VestasAnnualReport2025.pdf"
pages_of_interest = ((range(49, 51), "Risk Management"), (range(70, 74), "Material Risk"), (range(84, 92), "Climate Change"),(range(117, 118), "Cyber Security")) # Lesson learned - keep these in order, otherwise markdown extraction and secton reconstruction does not match

In [168]:
%%skip No need to show this anymore
doc_src = pymupdf.open(report_path)
#doc.metadata
doc_src[117].get_pixmap().pil_image()

skipping: No need to show this anymore


### Copy relevant pages from pdf

In [169]:
%%skip Only necessary once
doc_dst = pymupdf.open()

for (page_range, topic) in pages_of_interest:
    for page_num in page_range:
        page_src = doc_src[page_num]
        page_dst = doc_dst.new_page(-1, page_src.rect.width, page_src.rect.height)
        page_dst.show_pdf_page(page_src.rect, doc_src, page_num)

pp = pl.Path(doc_src.name).with_stem(pl.Path(doc_src.name).stem + "-extract")

doc_dst.save(pp,
    garbage=3,  # eliminate duplicate objects
    deflate=True,  # compress stuff where possible
)

skipping


### Load all pages of interest into markdown

In [124]:
pages = [page for (page_range, _) in pages_of_interest for page in page_range]
md = pymupdf4llm.to_markdown(doc_src, pages=pages, header=False, footer=False, page_chunks=True, show_progress=True)

# Reconstruct sectons
it = iter(md)
md_sections = [
    (section, page_range, list(itertools.islice(it, len(page_range))))
    for page_range, section in pages_of_interest
]

Parsing 15 pages of '/Users/timkaas/Projects/vestas-risk-case/data/VestasAnnualReport2025.pdf'...
[========================================] (15/15)
Generating markdown text...
[========================================] (15/15)


### Document splitting

Let’s test if splitting the document into chunks yields a higher recall to prevent a "lost-in-the-middle" effect. \
Potential pitfalls:
- Since the reports are converted from PDF to Markdown, the splits might be messed up due to incorrect headline parsing
- Splitting could lead to a loss of context if not split properly
- Splitting could carry a higher cost in terms of tokenization due to repeated prompt overhead.
- The same risk can be reported twice if split in context

Benefits:
- A split document can be ingested in parallel, speeding up the processing. However, I do not think speed is the main concern here

In [ ]:
splitter = MarkdownTextSplitter(chunk_size=400, chunk_overlap=20)
md_splits = splitter.create_documents([md[0]['text']])

md_splits
#md_header_splits = markdown_splitter.split_text(markdown_document)

In [126]:
def format_pages(pages) -> str:
    return "\n\n".join(
        f"=== PDF PAGE {page['metadata']['page_number']} ===\n\n{page['text']}"
        for page in pages
    )

def format_sections(sections):
    return "\n\n".join(f"# {section_name}\n\n{format_pages(pages)}" for section_name, page_rang, pages in sections)

In [ ]:
display(Markdown(format_sections(md_sections)))

In [137]:
headers_to_split_on = [
    ("#", "Header 1"),
    #("##", "Header 2"),
    #("###", "Header 3"),
]

#full_md = format_pages(md)
full_md = format_sections(md_sections)

#mdd = Document(page_content=md[0]['text'], metadata=md[0]['metadata'])

splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
#md_splits = sum((splitter.split_text(mdp['text']) for mdp in md), [])
md_splits = splitter.split_text(full_md)

current_page = []
# 3. Extract page numbers into metadata for each split
for doc in md_splits:
    # Find all page markers contained within this chunk
    found_pages = [int(p) for p in re.findall(r"=== PDF PAGE (\d+) ===", doc.page_content)]

    if found_pages:
        current_page = found_pages

    # Store page metadata
    doc.metadata["pages"] = current_page

In [139]:
[m.metadata for m in md_splits]
md_splits[0]

Document(metadata={'Header 1': 'Risk Management', 'pages': [50]}, page_content='# Risk Management  \n=== PDF PAGE 50 ===')

# Datamodels

In [60]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are an expert risk intelligence analyst. Extract all principal corporate risks from the provided annual report text with high precision."),
    ("human", "Section: {section_name}\nPages: {page_numbers}\n\nContent:\n{content}")
])

def build_extraction_chain(model_name: str = "gpt-4o", temperature: float = 0.0):
    llm = ChatOpenAI(model=model_name, temperature=temperature)
    structured_llm = llm.with_structured_output(ExtractionResult)
    return EXTRACTION_PROMPT | structured_llm

In [170]:
%%skip Test call to LLM
# Test invokation of a single section
llm_chain = build_extraction_chain()
section_name, page_range, pages = md_sections[0]
result = llm_chain.invoke({"section_name": section_name, "page_numbers": list(page_range), "content": format_pages(pages)})

skipping: Test call to LLM


In [148]:
batch_inputs = [
    {
        "section_name": section_name,
        "page_numbers": list(page_range),
        "content": format_pages(pages),
    }
    for section_name, page_range, pages in md_sections
]

results = llm_chain.batch(batch_inputs, config=RunnableConfig(max_concurrency=4))

In [156]:
#ExtractionResult(**(r.model_dump() for r in results))
risks = [rr.model_dump() for r in results for rr in r.risks]
result = ExtractionResult(risks=risks)

In [174]:
for r in risks:
    print(r)

{'title': 'Geopolitical and Regulatory Framework', 'description': 'Vestas faces significant challenges due to shifting geopolitical tensions and regulatory changes. Conflicts in regions like Ukraine and the Middle East disrupt stability, affecting global supply chains and operations. Trade tensions and assertive industrial policies in major economies further complicate the regulatory landscape, impacting market incentives and competition in the clean-tech sector.', 'category': <RiskCategory.GEOPOLITICAL: 'geopolitical'>, 'section': 'Main risks', 'pages': [51], 'mitigation': 'Vestas manages geopolitical risks by monitoring global developments, maintaining a global and regional manufacturing footprint, and implementing appropriate mitigations.', 'evidence': [{'page': 51, 'quote': 'In 2025, Vestas continued to navigate significant challenges arising from shifting geopolitical tensions across the world. Conflicts in Ukraine and the Middle East continued to disrupt regional stability, affec